# Human Training Notebook
It's a battle between the humans and the PPO models.
Use your local environment (not Google Colab) for this notebook

In [2]:

import os
import sys
# Get current notebook directory

current_dir = os.getcwd()

In [3]:
# Move up one level (to repo root)
repo_root = os.path.abspath(os.path.join(current_dir, ".."))
os.chdir(repo_root)
print("Working directory set to:", os.getcwd())

Working directory set to: /Users/nicole.r.frank/tiny-tackers


In [4]:
# Now set path for gym_sailing
sys.path.insert(0, "gym_sailing_environments/gym_sailing_gabo-tor")
import gym_sailing
print("Loaded from:", gym_sailing.__file__)

Loaded from: /Users/nicole.r.frank/tiny-tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py


In [5]:
import time
import numpy as np
import pandas as pd
import gymnasium as gym
import pygame

from stable_baselines3 import PPO

# Prevent gym-sailing quit() issue
import builtins
builtins.quit = lambda *args, **kwargs: None

/Users/nicole.r.frank/tiny-tackers/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [6]:
ENV_ID = "Sailboat-v0"
MAX_STEPS=10000

env = gym.make(ENV_ID, render_mode="human")
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
env.close()

Action space: Box(-1.0, 1.0, (1,), float64)
Observation space: Box([-10.          -3.14159265  -1.          -3.14159265   0.        ], [ 10.           3.14159265   1.           3.14159265 100.        ], (5,), float64)


In [7]:
def get_human_action(env):
    pygame.event.pump()
    keys = pygame.key.get_pressed()

    steer = 0.0

    if keys[pygame.K_LEFT]:
        steer = -1.0
    elif keys[pygame.K_RIGHT]:
        steer = 1.0

    action = np.array([steer], dtype=np.float32)

    # Keep action inside env limits
    action = np.clip(action, env.action_space.low, env.action_space.high)

    return action

In [ ]:
def run_human_episode(env_id=ENV_ID, max_steps=MAX_STEPS):
    pygame.init()
    env = gym.make(env_id, render_mode="human")
    obs, info = env.reset()

    total_reward = 0.0
    timesteps = 0
    running = True

    print("Human controls:")
    print("LEFT arrow = steer left")
    print("RIGHT arrow = steer right")
    print("ESC = quit episode")
    print("Press SPACE to start...")

    # --- WAIT FOR SPACE ---
    waiting = True

    # small delay so window appears before input
    time.sleep(0.5)

    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                waiting = False
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    waiting = False

        pygame.event.pump()
        time.sleep(0.01)

    # --- MAIN GAME LOOP ---
    while running and timesteps < max_steps:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False

        action = get_human_action(env)
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        timesteps += 1

        if terminated or truncated:
            break

        time.sleep(0.03)

    env.close()
    pygame.quit()

    return {
        "agent": "Human",
        "total_reward": total_reward,
        "timesteps": timesteps
    }

In [9]:
MODEL_PATH = "models/ppo/base/ppo_base_1M.zip"

model = PPO.load(MODEL_PATH)
print("Loaded model:", MODEL_PATH)

Loaded model: models/ppo/base/ppo_base_1M.zip


In [12]:
import pygame
import gymnasium as gym
import time

def run_ppo_episode(model, env_id=ENV_ID, max_steps=MAX_STEPS):
    # Let Gymnasium handle the display initialization via render_mode
    env = gym.make(env_id, render_mode="human")
    obs, info = env.reset()

    total_reward = 0.0
    timesteps = 0
    running = True

    print("PPO agent running... ESC = quit episode")

    try:
        while running and timesteps < max_steps:
            # Handle Pygame events so the window stays responsive
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                    running = False

            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)

            total_reward += reward
            timesteps += 1

            if terminated or truncated:
                break

            time.sleep(0.03)

    finally:
        # This is the most important part!
        env.close() 
        # Explicitly shut down pygame's display and engine
        pygame.display.quit()
        pygame.quit()

    return {
        "agent": "PPO Base Model",
        "total_reward": total_reward,
        "timesteps": timesteps
    }

In [10]:
import builtins
builtins.quit = lambda *args, **kwargs: None

human_score = run_human_episode()
human_score

{'agent': 'Human',
 'total_reward': np.float64(386.34601148438173),
 'timesteps': 936}

In [13]:
import builtins
builtins.quit = lambda *args, **kwargs: None

ppo_score = run_ppo_episode(model)

ppo_score

PPO agent running... ESC = quit episode


error: video system not initialized

In [ ]:
results = pd.DataFrame([human_score, ppo_score])
results["winner_by_reward"] = results["total_reward"] == results["total_reward"].max()

winner = results.loc[results["total_reward"].idxmax()]

display(results)

print(f"Winner: {winner['agent']}")
print(f"Reward: {winner['total_reward']:.2f}")
print(f"Timesteps: {winner['timesteps']}")